In [1]:
import torch
import torch.nn as nn

In [2]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.conv(x)

In [3]:
block = DoubleConv(3, 64)

x = torch.randn(1, 3, 512, 512)

y = block(x)

print("Input Shape :", x.shape)
print("Output Shape:", y.shape)

Input Shape : torch.Size([1, 3, 512, 512])
Output Shape: torch.Size([1, 64, 512, 512])


In [4]:
class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):

        skip = self.conv(x)
        down = self.pool(skip)

        return skip, down

In [5]:
encoder = Down(3, 64)

x = torch.randn(1, 3, 512, 512)

skip, down = encoder(x)

print("Skip Shape :", skip.shape)
print("Down Shape :", down.shape)

Skip Shape : torch.Size([1, 64, 512, 512])
Down Shape : torch.Size([1, 64, 256, 256])


In [6]:
bottleneck = DoubleConv(64, 128)

x = torch.randn(1, 64, 256, 256)

y = bottleneck(x)

print("Input Shape :", x.shape)
print("Output Shape:", y.shape)

Input Shape : torch.Size([1, 64, 256, 256])
Output Shape: torch.Size([1, 128, 256, 256])


In [7]:
class Up(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

    def forward(self, x):
        return self.up(x)

In [8]:
up = Up(128, 64)

x = torch.randn(1, 128, 256, 256)

y = up(x)

print("Input Shape :", x.shape)
print("Output Shape:", y.shape)

Input Shape : torch.Size([1, 128, 256, 256])
Output Shape: torch.Size([1, 64, 512, 512])


In [9]:
class Decoder(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(out_channels * 2, out_channels)

    def forward(self, x, skip):

        x = self.up(x)

        # Concatenate skip connection
        x = torch.cat([skip, x], dim=1)

        x = self.conv(x)

        return x

In [10]:
decoder = Decoder(128, 64)

# Bottleneck output
x = torch.randn(1, 128, 256, 256)

# Skip connection from encoder
skip = torch.randn(1, 64, 512, 512)

y = decoder(x, skip)

print("Decoder Output Shape:", y.shape)

Decoder Output Shape: torch.Size([1, 64, 512, 512])
